In [2]:
import torch
from corpus import TaggedCorpus
from hmm import HiddenMarkovModel
from pathlib import Path

corpus = TaggedCorpus(Path("../data/icsup"))
model = HiddenMarkovModel(corpus.tagset, corpus.vocab)

# 设置一些简单的counts
model.A_counts = torch.tensor([
    [10.0, 5.0, 85.0, 0.0],  # C -> C, H, EOS, BOS
    [47.0, 9.0, 3.0, 0.0],   # H -> C, H, EOS, BOS  
    [0.0, 0.0, 0.0, 0.0],    # EOS -> (none)
    [1.0, 99.0, 0.0, 0.0],   # BOS -> C, H, EOS, BOS
])


model.B_counts = torch.tensor([
    [59.0, 1.0, 0.0],   # C -> 1, 2, 3
    [0.0, 1.0, 57.0],   # H -> 1, 2, 3
    [0.0, 0.0, 0.0],    # EOS
    [0.0, 0.0, 0.0],    # BOS
])
model.M_step(λ=0)

print("A matrix:")
print(model.A)
print("\nRow sums:")
print(model.A.sum(dim=1))

# 应该输出接近[1, 1, 0, 1]

A matrix:
tensor([[0.1000, 0.0500, 0.8500, 0.0000],
        [0.7966, 0.1525, 0.0508, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000],
        [0.0100, 0.9900, 0.0000, 0.0000]])

Row sums:
tensor([1., 1., 0., 1.])


In [3]:
model.printAB()

Transition matrix A:
	C	H	_EOS_TAG_	_BOS_TAG_
C	0.100	0.050	0.850	0.000
H	0.797	0.153	0.051	0.000
_EOS_TAG_	0.000	0.000	0.000	0.000
_BOS_TAG_	0.010	0.990	0.000	0.000

Emission matrix B:
	1	2	3
C	0.983	0.017	0.000
H	0.000	0.017	0.983
_EOS_TAG_	0.000	0.000	0.000
_BOS_TAG_	0.000	0.000	0.000




In [5]:
import torch
from corpus import TaggedCorpus
from hmm import HiddenMarkovModel
from pathlib import Path

icsup = TaggedCorpus(Path("../data/icsup"))
model = HiddenMarkovModel(icsup.tagset, icsup.vocab)

# 训练一个iteration
model._zero_counts()
for sentence in icsup:
    isent = model._integerize_sentence(sentence, icsup)
    model.E_step(isent)

print("A_counts after E-step:")
print(model.A_counts)
print("\nA_counts row sums:")
print(model.A_counts.sum(dim=1))

# 如果这些counts看起来很奇怪（比如负数、太大、或分布不合理），问题在E_step
# 如果counts正常，问题在M_step或其他地方

A_counts after E-step:
tensor([[16.0000,  2.0000,  2.0000,  0.0000],
        [ 2.0000, 16.0000,  2.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 2.0000,  2.0000,  0.0000,  0.0000]])

A_counts row sums:
tensor([20.0000, 20.0000,  0.0000,  4.0000])
